In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"

for name in ["listings_clean.csv", "reviews_monthly.csv", "listing_sentiment.csv"]:
    df = pd.read_csv(DATA / name)
    print("=" * 70)
    print(f"{name}   shape={df.shape}")
    print("-" * 70)
    print(df.dtypes.to_string())
    print("-" * 70)
    print(df.head(3).to_string())
    print()

listings_clean.csv   shape=(92638, 31)
----------------------------------------------------------------------
id                             int64
host_id                        int64
property_type                 object
room_type                     object
accommodates                   int64
bedrooms                     float64
beds                         float64
neighbourhood_cleansed        object
latitude                     float64
longitude                    float64
price                        float64
hosts_time_as_host_years     float64
host_is_superhost             object
host_listings_count          float64
minimum_nights               float64
availability_365               int64
number_of_reviews              int64
reviews_per_month            float64
first_review                  object
last_review                   object
estimated_occupancy_l365d      int64
estimated_revenue_l365d      float64
review_scores_rating         float64
review_scores_cleanliness    float64
re

## Dashboard Justification 

### Inclusions

The ‌dashboard ‌uses ‌a star schema. Two tables sit on the filtering side, `listings_clean` and `DimDate`, and two sit as facts, `reviews_monthly` and `listing_sentiment`. Every relationship is one-to-many and single-direction. I picked this setup on purpose instead of pushing everything into one flat table, mainly so borough and room-type slicers behave the same way across the whole report, sentiment visuals included.

There are nine DAX measures. They cover the key acquisition numbers, Total Est Revenue, Median Price, Median Occupancy, and Superhost %. On top of that, two time-intelligence measures, Reviews LY and Reviews YoY %, were built off a complete 2009–2026 date table, not just the snapshot period. The set also includes a Borough Revenue Rank built with RANKX, which recalculates cleanly in normal filter context and still holds up under row-level security.

The report is split across four pages. Executive Overview is the quick, ten-second read. Borough & Property Analysis digs into borough and room-type patterns, including a price vs room-type heatmap and the price-revenue scatter. Revenue & Guest Experience brings the superhost occupancy gap together with the sentiment results. Then there’s a hidden drill-through page, Borough Detail, so you can right-click into a single borough and see the whole story, including where it sits in the ranking against the other 32.

On the Revenue & Guest Experience page, I added the Key Influencers AI visual. It looks at `estimated_occupancy_l365d` alongside host and property characteristics. The point is to let stakeholders ask “what’s driving demand?” in the report itself, instead of only scanning fixed charts.

Row-level security is set up through two roles, Central London Analyst and Outer London Analyst. Both filter on `listings_clean[Borough]`, which means the same dashboard can go out to different regional teams while keeping boroughs outside their scope out of view.


### Exclusions

Any ‌fields ‌that ‌come from reviews, starting with `review_count` and including anything that depends on it like `estimated_revenue_l365d`, were intentionally left out of the Key Influencers inputs. In the raw dataset, occupancy is already calculated off review counts (r = 0.74). If those variables were fed into the visual, it would happily “find” the loop and frame it as a meaningful driver, when it’s really the same circular leakage we spotted earlier in the project.

A Decomposition Tree was on the table as a different AI visual. It was dropped in favor of Key Influencers because the stakeholder isn’t asking “how does total revenue split across dimensions,” they’re asking “what pushes occupancy higher.” Key Influencers lines up with that decision-focused question.

For both price and occupancy, medians were used everywhere instead of averages. The distributions are strongly right-skewed, for example, the top 10% of listings account for 43.2% of revenue, so a mean would paint an inflated picture of what a typical acquisition looks like.

A What-If pricing knob was also discussed, but it didn’t make it into this version. The observed link between price and revenue is only modest (r = 0.35), so a slider that suggests “pick a price, receive this revenue” would sell the model as more predictive than it really is, and add less decision value than it appears to.

### Deprioritised stakeholder

This ‌dashboard ‌is ‌meant for the Head of Portfolio, specifically for acquisition and pricing calls. It intentionally puts an Operations or Guest Experience manager second, because they’d need a completely different product: a live, listing-by-listing view that routes complaints as they come in, built around NLP topics like “hot water” or “didn’t work,” not a borough-level sentiment chart pulled from a fixed snapshot and refreshed on a schedule. To build the ops version, you’d need real-time review intake plus integration with ticketing systems, and that sits outside the scope of a portfolio acquisition dashboard. Sentiment is shown here only as far as it helps with a buy, hold, or sell decision, not for running day-to-day guest experience response.